# Interactuando con Python, Pandas y Cloud SQL - PostgreSQL

Instalamos lo necesario

In [18]:
!pip install psycopg2-binary
!pip install pandas
!pip install sqlalchemy

Permitimos a Colab que interactue con GCP

In [2]:
from google.colab import auth
auth.authenticate_user()

Importamos librerías

In [20]:
import pandas as pd
import json
import requests
from sqlalchemy import create_engine

Tomaremos como ejemplo los archivos schema.json y part-00000

Definimos el nombre de las columnas en nuestro archivo

In [4]:
schemas = json.load(open('./schemas.json'))
columns = [col['column_name'] for col in sorted(schemas['orders'], key=lambda col: col['column_position'])]
columns

['order_id', 'order_date', 'order_customer_id', 'order_status']

Vista previa de nuestro archivo

In [5]:
orders = pd.read_csv('./part-00000', names=columns)
orders

,order_id,order_date,order_customer_id,order_status
0,1,2013-07-25 00:00:00.0,11599,CLOSED
1,2,2013-07-25 00:00:00.0,256,PENDING_PAYMENT
2,3,2013-07-25 00:00:00.0,12111,COMPLETE
3,4,2013-07-25 00:00:00.0,8827,CLOSED
4,5,2013-07-25 00:00:00.0,11318,COMPLETE
...,...,...,...,...
68878,68879,2014-07-09 00:00:00.0,778,COMPLETE
68879,68880,2014-07-13 00:00:00.0,1117,COMPLETE
68880,68881,2014-07-19 00:00:00.0,2518,PENDING_PAYMENT
68881,68882,2014-07-22 00:00:00.0,10000,ON_HOLD


Query/transformación de los datos

In [6]:
daily_status_count = orders. \
    groupby(['order_date', 'order_status'])['order_id']. \
    agg(order_count='count'). \
    reset_index()
daily_status_count

,order_date,order_status,order_count
0,2013-07-25 00:00:00.0,CANCELED,1
1,2013-07-25 00:00:00.0,CLOSED,20
2,2013-07-25 00:00:00.0,COMPLETE,42
3,2013-07-25 00:00:00.0,ON_HOLD,5
4,2013-07-25 00:00:00.0,PAYMENT_REVIEW,3
...,...,...,...
3198,2014-07-24 00:00:00.0,PAYMENT_REVIEW,2
3199,2014-07-24 00:00:00.0,PENDING,23
3200,2014-07-24 00:00:00.0,PENDING_PAYMENT,37
3201,2014-07-24 00:00:00.0,PROCESSING,20


Definimos los parámetros de conexión a nuestra base de datos

In [7]:
host = '34.23.43.163'
port = 5432
database = 'test'
user = 'dyanez'
password = '0?E@oFpqCA4uDuE='

In [21]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=user,
    password=password,
    host="34.23.43.163",
    port=5432,
    database=database
)

engine = create_engine(connection_url)

Para que Colab pueda conectarse, necesitamos añadir su IP en la consola o con gcloud:
```gcloud sql instances patch sd-local-sql-pg-1 --authorized-networks <ip-addres>```

In [10]:
import requests
ip = requests.get('https://api.ipify.org').text
print(f'Current IP Address: {ip}')

Current IP Address: 136.119.31.87


Una vez configurado, insertamos la data en nuestra base de datos.

In [24]:
daily_status_count.to_sql(
    'daily_status_count',
    engine,
    if_exists='replace',
    index=False
)

203

La leemos

In [25]:
dsc = pd.read_sql(
    'daily_status_count',
    engine
)
dsc

,order_date,order_status,order_count
0,2013-07-25 00:00:00.0,CANCELED,1
1,2013-07-25 00:00:00.0,CLOSED,20
2,2013-07-25 00:00:00.0,COMPLETE,42
3,2013-07-25 00:00:00.0,ON_HOLD,5
4,2013-07-25 00:00:00.0,PAYMENT_REVIEW,3
...,...,...,...
3198,2014-07-24 00:00:00.0,PAYMENT_REVIEW,2
3199,2014-07-24 00:00:00.0,PENDING,23
3200,2014-07-24 00:00:00.0,PENDING_PAYMENT,37
3201,2014-07-24 00:00:00.0,PROCESSING,20


Probamos otro query

In [26]:
dsc = pd.read_sql(
    '''
        SELECT order_status, sum(order_count) AS order_count FROM daily_status_count
        GROUP BY 1
        ORDER BY 2 DESC
    ''',
    engine
)
dsc

,order_status,order_count
0,COMPLETE,22899.0
1,PENDING_PAYMENT,15030.0
2,PROCESSING,8275.0
3,PENDING,7610.0
4,CLOSED,7556.0
5,ON_HOLD,3798.0
6,SUSPECTED_FRAUD,1558.0
7,CANCELED,1428.0
8,PAYMENT_REVIEW,729.0
